# Autoencoder Architecture

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

# Define the path to the zip file and the extraction directory
zip_file_path = '/content/drive/MyDrive/UCSD_Anomaly_Dataset.zip'
extract_path = '/content/UCSD_Anomaly'

# Extract the zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed.")

Extraction completed.


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm import tqdm

In [ ]:
# Define the dataset class
class UCSDDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.frames = []
        self.load_frames()

    def load_frames(self):
        for folder in os.listdir(self.root_dir):
            folder_path = os.path.join(self.root_dir, folder)
            if os.path.isdir(folder_path) and (folder.startswith('Train') or folder.startswith('Test')):
                for frame_name in os.listdir(folder_path):
                    if frame_name.endswith('.tif'):
                        self.frames.append(os.path.join(folder_path, frame_name))
        print(f"Loaded {len(self.frames)} frames from {self.root_dir}")

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        frame_path = self.frames[idx]
        image = Image.open(frame_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

# Transformations for the images
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# Create datasets and dataloaders
train_dataset = UCSDDataset('/content/UCSD_Anomaly/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Train', transform=transform)
test_dataset = UCSDDataset('/content/UCSD_Anomaly/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Test', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

# Check if data loading works correctly
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Loaded 6800 frames from /content/UCSD_Anomaly/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Train
Loaded 7200 frames from /content/UCSD_Anomaly/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Test
Number of training samples: 6800
Number of testing samples: 7200


In [ ]:
# Define the autoencoder class with ResNet-50 backbone
class ResNetAutoencoder(nn.Module):
    def __init__(self):
        super(ResNetAutoencoder, self).__init__()
        # Encoder using ResNet-50
        resnet = models.resnet50(pretrained=True)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(2048, 1024, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(1024, 512, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 3, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetAutoencoder().to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for data in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs = data.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)

    train_loss = train_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}")

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 180MB/s]
Epoch 1/20: 100%|██████████| 213/213 [00:57<00:00,  3.68it/s]


Epoch 1/20, Loss: 0.0144


Epoch 2/20: 100%|██████████| 213/213 [00:57<00:00,  3.69it/s]


Epoch 2/20, Loss: 0.0034


Epoch 3/20: 100%|██████████| 213/213 [00:58<00:00,  3.65it/s]


Epoch 3/20, Loss: 0.0024


... (30 outputs left out)

Epoch 19/20: 100%|██████████| 213/213 [01:00<00:00,  3.53it/s]


Epoch 19/20, Loss: 0.0004


Epoch 20/20: 100%|██████████| 213/213 [01:00<00:00,  3.54it/s]

Epoch 20/20, Loss: 0.0004


In [ ]:
from PIL import Image
# Compute reconstruction errors on the test set
def compute_reconstruction_error(model, data_loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for data in data_loader:
            inputs = data.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            errors.extend(loss.cpu().numpy().flatten())
    return errors

# Compute reconstruction errors on the test set
test_errors = compute_reconstruction_error(model, test_loader)

# Assuming ground truth labels for anomalies (0 for normal, 1 for anomaly)
# For this example, we will create a dummy ground truth labels array.
# Replace this with actual labels if available.
test_labels = np.zeros(len(test_errors))
# Mark some frames as anomalous for demonstration purposes
test_labels[len(test_labels) // 2:] = 1

# Calculate ROC-AUC
roc_auc = roc_auc_score(test_labels, test_errors)
print(f"ROC-AUC: {roc_auc:.4f}")

# Calculate EER
fpr, tpr, thresholds = roc_curve(test_labels, test_errors)
fnr = 1 - tpr
eer_threshold = thresholds[np.nanargmin(np.absolute(fnr - fpr))]
eer = fpr[np.nanargmin(np.absolute(fnr - fpr))]
print(f"EER: {eer:.4f}")

OSError: -2

# GAN Autoencoder Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Hyperparameters
latent_size = 100
batch_size = 32
lr = 0.0002
epochs = 20

# Image processing
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

In [ ]:
from PIL import Image
import os
import torch
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.frames = self._load_frames()

    def _load_frames(self):
        frames = []
        for folder in os.listdir(self.root_dir):
            folder_path = os.path.join(self.root_dir, folder)
            if os.path.isdir(folder_path):
                for file in os.listdir(folder_path):
                    file_path = os.path.join(folder_path, file)
                    if file.endswith('.tif'):
                        frames.append(file_path)
        return frames

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        frame_path = self.frames[idx]
        # Add a try-except block to catch potential errors when opening images
        try:
            image = Image.open(frame_path).convert("RGB")
            if self.transform:
                image = self.transform(image)  # Apply transform if available
            else:
                image = transforms.ToTensor()(image)  # Convert to tensor if no transform provided
            return image
        except Exception as e:
            print(f"Error loading image {frame_path}: {e}")
            return None  # Return None for failed images


# Initialize dataset and dataloader, removing multiprocessing for now
train_dataset = CustomDataset('/content/UCSD_Anomaly/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Train', transform=transforms.ToTensor())  # Remove transform for simplicity
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)  # Set num_workers to 0

# Iterate through the dataloader and check for errors
for i, data in enumerate(train_loader):
    if data is None:
        print(f"Encountered None data at index {i}")
    else:
        print(f"Batch {i} loaded successfully with shape {data.shape}")

test_dataset = CustomDataset('/content/UCSD_Anomaly/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped1/Test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=1, pin_memory=False)

Batch 0 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 1 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 2 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 3 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 4 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 5 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 6 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 7 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 8 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 9 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 10 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 11 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 12 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 13 loaded successfully with shape torch.Size([32, 3, 158, 238])
Batch 14 loaded successfully w

In [ ]:
import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, latent_size=100, num_channels=3):
        super(Generator, self).__init__()
        self.latent_size = latent_size
        self.num_channels = num_channels

        self.fc = nn.Linear(latent_size, 512 * 8 * 8)
        self.conv1 = nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(256)
        self.conv2 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)  # Another upsampling layer
        self.bn4 = nn.BatchNorm2d(32)
        self.conv5 = nn.ConvTranspose2d(32, num_channels, kernel_size=4, stride=2, padding=1)  # Final upsampling

        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, x):
        batch_size = x.size(0)
        x = self.fc(x)
        x = x.view(batch_size, 512, 8, 8)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.conv4(x)  # Pass through the new ConvTranspose2d layer
        x = self.bn4(x)
        x = self.relu(x)
        x = self.conv5(x)  # Final upsampling
        x = self.tanh(x)
        return x

# Example usage
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_size = 100
generator = Generator(latent_size=latent_size).to(device)

# Generate fake data
batch_size = 32
noise = torch.randn(batch_size, latent_size, device=device)  # Remove unnecessary dimensions
fake_data = generator(noise)
print(fake_data.shape)  # Check the shape of the generated fake data


torch.Size([32, 3, 256, 256])


In [ ]:
import torch
import torch.nn as nn

class Discriminator(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, kernel_size=4, stride=2, padding=1, height=256, width=256):
        super(Discriminator, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, base_channels, kernel_size, stride, padding)
        self.instance_norm1 = nn.InstanceNorm2d(base_channels)
        self.leaky_relu1 = nn.LeakyReLU(0.2, inplace=True)

        self.conv2 = nn.Conv2d(base_channels, base_channels * 2, kernel_size, stride, padding)
        self.instance_norm2 = nn.InstanceNorm2d(base_channels * 2)
        self.leaky_relu2 = nn.LeakyReLU(0.2, inplace=True)

        self.conv3 = nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size, stride, padding)
        self.instance_norm3 = nn.InstanceNorm2d(base_channels * 4)
        self.leaky_relu3 = nn.LeakyReLU(0.2, inplace=True)

        self.conv4 = nn.Conv2d(base_channels * 4, base_channels * 8, kernel_size, stride, padding)
        self.instance_norm4 = nn.InstanceNorm2d(base_channels * 8)
        self.leaky_relu4 = nn.LeakyReLU(0.2, inplace=True)

        self.flatten = nn.Flatten()

        # Calculate the output dimensions of the last conv layer
        convw = width // (2 ** 4)
        convh = height // (2 ** 4)
        self.fc1 = nn.Linear(base_channels * 8 * convw * convh, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.conv1(x)
        x = self.instance_norm1(x)
        x = self.leaky_relu1(x)

        x = self.conv2(x)
        x = self.instance_norm2(x)
        x = self.leaky_relu2(x)

        x = self.conv3(x)
        x = self.instance_norm3(x)
        x = self.leaky_relu3(x)

        x = self.conv4(x)
        x = self.instance_norm4(x)
        x = self.leaky_relu4(x)

        x = self.flatten(x)

        x = self.fc1(x)
        x = self.sigmoid(x)
        return x

# Example usage
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use the actual height and width of your input images
height = 256  # Replace with the actual height
width = 256   # Replace with the actual width
discriminator = Discriminator(height=height, width=width).to(device)

In [ ]:
import torchvision
import torchvision.transforms as transforms

import torch.nn.init as init

# Define a weight initialization function
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        init.normal_(m.weight.data, 1.0, 0.02)
        init.constant_(m.bias.data, 0)

# Initialize models
generator = Generator(latent_size=latent_size).to(device)
discriminator = Discriminator(height=256, width=256).to(device)

# Initialize weights
generator.apply(weights_init)
discriminator.apply(weights_init)

# Loss function
criterion = nn.BCELoss()

# Optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))


In [ ]:
for epoch in range(epochs):
    for i, data in enumerate(train_loader, 0):
        real_data = data.to(device)
        batch_size = real_data.size(0)
        label_real = torch.full((batch_size,), 1.0, device=device)
        label_fake = torch.full((batch_size,), 0.0, device=device)

        ### Train Discriminator ###
        optimizer_D.zero_grad()

        # Real data
        output_real = discriminator(real_data).view(-1)
        errD_real = criterion(output_real, label_real)
        errD_real.backward()

        # Fake data
        noise = torch.randn(batch_size, latent_size, device=device)
        fake_data = generator(noise)
        output_fake = discriminator(fake_data.detach()).view(-1)
        errD_fake = criterion(output_fake, label_fake)
        errD_fake.backward()

        # Combine discriminator losses
        errD = errD_real + errD_fake
        optimizer_D.step()

        ### Train Generator ###
        optimizer_G.zero_grad()

        output = discriminator(fake_data).view(-1)
        errG = criterion(output, label_real)
        errG.backward()

        optimizer_G.step()

        if i % 10 == 0:
            print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f'
                  % (epoch, epochs, i, len(train_loader),
                     errD.item(), errG.item()))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x64512 and 131072x1)